In [ ]:
import pandas as pd

# Path to data frame containing all WSIs with a matched rekvnr as well as SNOMED categories
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)

print(df_all.head())

In [ ]:
from helper_functions import convert_and_flatten

df_all = convert_and_flatten(df_all, ["wsi filenames", "T", "M", "Other", "T category", "M category"])
print("Dataframe shape: ", df_all.shape)


In [ ]:
# Flatten the lists of filenames into a single list
all_filenames = [fname for sublist in df_all['wsi filenames'] for fname in sublist]

# Get unique filenames
unique_wsi = set(all_filenames)

# Check WSI count matches number of WSI files (wsi count, wsi filenames)
print("Sum of wsi count: ", df_all["wsi count"].sum())
print("Number of unique wsi filenames: ",  len(unique_wsi))

In [ ]:
df_all.describe()

In [ ]:
# Cross-tabulations for categorical variables:
pd.crosstab(df_all['team'], df_all['sex'])

In [ ]:
# Distribution of material types
import matplotlib.pyplot as plt
import seaborn as sns

# Type of material, mattype, mattype tekst
mattype_counts = df_all['mattype tekst'].value_counts()
print('Material types:\n', mattype_counts)

plt.figure(figsize=(10,6))
mattype_counts.plot(kind='bar')
plt.title("Material Types (mattype tekst)")
plt.xlabel("Material Type")
plt.ylabel("Number of Cases")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Material count
plt.figure(figsize=(12,6))
sns.boxplot(data=df_all, x='mattype tekst', y='matantal')
plt.title("Distribution of Specimen Count (matantal) for Material Types")
plt.xlabel("Material Type")
plt.ylabel("Specimen Count")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
cols_categorical = ["team", "sex", "alder gruppe", "mattype"]
cols_dates = ["modtdato", "rekvdato"]
cols_numerical = ["alder", "matantal","wsi count"]

In [ ]:
# Check unique counts for categorical variables
for col in cols_categorical:
    print(df_all[col].value_counts())
    print("-"*40)

In [ ]:
# Check normalized categorical distributions to identify class imbalance
for col in cols_categorical: 
    print(df_all[col].value_counts(normalize=True))
    print("-"*40)

In [ ]:
# Histograms for numeric distributions
import matplotlib.pyplot as plt

for col in cols_numerical: 
    plt.figure(figsize=(6, 4))
    df_all[col].dropna().hist(bins=30)
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.grid(False)
    plt.show()

In [ ]:
# Barplots for categorical distributions
for col in cols_categorical:
    plt.figure(figsize=(6, 4))
    value_counts = df_all[col].value_counts(dropna=False)
    plt.bar(value_counts.index.astype(str), value_counts.values)
    plt.title(f"Barplot of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:
# Patient Demographics
# Sex distribution
print("Sex distribution:\n", df_all['sex'].value_counts())

# Age distribution
plt.figure(figsize=(8,5))
df_all['alder'].hist(bins=30, color='skyblue', edgecolor='black')
plt.title('Age Distribution (All Patients)')
plt.xlabel('Age')
plt.ylabel('Number of Patients')
plt.show()

# Age distribution by sex
plt.figure(figsize=(8,5))
df_all[df_all['sex']=='M']['alder'].hist(bins=30, alpha=0.6, label='Male', color='lightblue', edgecolor='black')
df_all[df_all['sex']=='F']['alder'].hist(bins=30, alpha=0.6, label='Female', color='pink', edgecolor='black')
plt.title('Age Distribution by Sex')
plt.xlabel('Age')
plt.ylabel('Number of Patients')
plt.legend()
plt.show()

In [ ]:
# FIGURE 1
# Bar plot showing the number of distinct codes per prefix

from helper_functions import to_tuples

df_all = to_tuples(df_all)

# Combine all codes into a single list
all_codes = pd.concat([df_all['M'], df_all['T'], df_all['Other']]).dropna().unique()
all_codes = [code for c in all_codes for code in c]
prefixes = [code[0] for code in all_codes if code]
counts = pd.Series(prefixes).value_counts()

plt.figure(figsize=(6,4))
plt.bar(counts.index, counts.values, color='steelblue')
plt.xlabel('Code Prefix')
plt.ylabel('Number of Distinct Codes')
plt.title('Number of Distinct Codes per Prefix')
plt.tight_layout()
plt.show()

In [ ]:
from snomed_hierarchy import SNOMEDCodes

# Path to SNOMED codes (all codes with code history)
snomed_path = "D:/DATA/patoSnoMed_2025-04.xlsx"

xls_snomed = pd.read_excel(snomed_path)
df_snomed = pd.DataFrame(xls_snomed, columns=['SKSkode', 'Kodetekst'])

snomed = SNOMEDCodes(df_snomed)
snomed_code_dict = snomed.code_to_text()


In [ ]:
from helper_functions import MetadataExplorer

code_explorer = MetadataExplorer(df_all, snomed_code_dict)

In [ ]:
# Plot based on actual codes (by_category = False) or snomed categories (by_category = True)
# Subset can be selected using, subset_col (e.g. sex) and subset_name (e.g. F)
code_explorer.plot_top_counts(letter = "T", n = 10, by_category = True)

In [ ]:
# Plot based on actual codes (by_category = False) or snomed categories (by_category = True)
code_explorer.plot_many(subset_col = "sex", letter = "T", n = 10, by_category = True)

In [ ]:
# FIGURE 2
# Top morphological codes (excluding “normal” categories)
# Horizontal bar plot showing the ten most common morphologic diagnoses such as leiomyoma, chronic inflammation and squamous‑cell carcinoma. 
# This highlights the diversity beyond the dominant screening cases.

# Remove most common categories of normal morphology
exclude_texts = ['normalt væv', 'ingen tegn på malignitet', 'resektionsrande frie']

code_explorer.plot_top_counts(letter = "M", n = 10, by_category = False, exclude_text = exclude_texts, text = "(excl. normal categories)")

In [ ]:
code_explorer.plot_many(subset_col = "alder gruppe", letter = "M", n = 10, by_category = False, exclude_text=exclude_texts)

In [ ]:
code_explorer.plot_top_counts(letter = "T", n = 10, by_category = True)
code_explorer.plot_top_counts(letter = "M", n = 10, by_category = True)